# Notebook 3 — Downstream detector training

Business-value check for Notebook 2's accepted synth:

**fine-tune YOLOv8n** on **real-only** vs **real + synthetic**, evaluate both on the **same held-out real test**.

### Will synth help?

COCO-pretrained YOLOv8n has **no** `traffic_cone` / `trash_bin` classes — we fine-tune a 2-class head on a strong backbone.
Synth *can* lift rare-class AP when real rares are scarce, but YOLO-World auto-labels are noisy, so results are not guaranteed.
Either outcome is useful teaching: measure, don't assume.

Ablations use **50% / 100% of whatever usable synth Notebook 2 accepted per class** (not a hardcoded 50/100).

Depends on [02_batch_dataset_generation.ipynb](02_batch_dataset_generation.ipynb) export under `outputs/<dataset>/nb2/`.

---
## What are we measuring? (detection metrics cheat sheet)

This notebook trains an **object detector**: for each image it must output **boxes** (where?) and **class labels** (what?).

| Term | Plain English |
|------|----------------|
| **IoU** (Intersection over Union) | Overlap between predicted box and ground-truth box. 1.0 = perfect overlap; 0 = no overlap. |
| **AP50** (Average Precision @ IoU 0.5) | Per-class score: “How good are we at finding this class if we only require **roughly** the right location?” A prediction counts as correct when class is right **and** IoU ≥ 0.5. |
| **mAP50** | **Mean** AP50 across all classes — one headline number for overall detection quality. |
| **mAP50-95** | Stricter version: average AP across IoU thresholds from 0.5 to 0.95. Penalizes sloppy boxes. Usually much lower than mAP50. |
| **Precision / Recall** | Of all predictions, how many were right? / Of all real objects, how many did we find? |

**How to read your table:** focus on **per-class AP50** for the rare classes (`traffic_cone`, `trash_bin`). Overall mAP50 can be dragged down by empty background images. Absolute values are often low on tiny datasets — look for **relative lift** (`real_synth_50` / `real_synth_100` vs `real_only`).

Want something more intuitive? See [03.5_classification_training_and_evaluation.ipynb](03.5_classification_training_and_evaluation.ipynb) — same NB2 export, but **one label per image** (accuracy / F1 instead of boxes).

---
## 0. Setup

In [ ]:
import sys
from pathlib import Path

%matplotlib inline

def _find_project_root() -> Path:
    here = Path.cwd().resolve()
    search = [here, *here.parents]
    for base in list(search):
        nested = base / "implementations" / "edge_case_image_generation"
        if nested.is_dir():
            search.append(nested)
    for base in search:
        if (base / "src" / "edgecase_synthesis").is_dir() and (base / "configs").is_dir():
            return base
    raise FileNotFoundError("Could not find edge_case_image_generation root")


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("PROJECT_ROOT =", PROJECT_ROOT)

---
## 1. Knobs

Class names, aliases (Mapillary GT + YOLO-World synonyms), and train budget live here — not in package code.

In [ ]:
import os

os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

from edgecase_synthesis.config import load_config
from edgecase_synthesis.detection_train import load_manifest

# --- learner knobs ---------------------------------------------------------
DATASET = "mapillary_vistas"
HARDWARE = "gpu_l4"  # or "gpu_l4x2" / "cpu" for smoke run

CLASS_NAMES = ["traffic_cone", "trash_bin"]
# Mapillary stems + YOLO-World query phrases from anomaly YAMLs.
LABEL_ALIASES = {
    "traffic_cone": ["traffic cone", "orange cone", "cone"],
    "trash_bin": ["trash can", "garbage bin", "waste bin", "dumpster"],
}

MODEL = "yolov8n.pt"
EPOCHS = 40          # ~5–15 min on L4 for this toy set; raise if underfitting
IMGSZ = 640
BATCH = 16           # drop to 8 if VRAM is tight
PATIENCE = 15
SEED = 42

# Cap empty `scene` backgrounds in the YOLO train set (NB2 still keeps full
# scene pool for synth seeds). ~500 scene / ~50 rare is too imbalanced for OD.
MAX_SCENE_TRAIN = 150

NB2_DIR_NAME = "nb2"  # outputs/<dataset>/nb2 from Notebook 2
# ---------------------------------------------------------------------------

cfg = load_config(
    start=PROJECT_ROOT,
    overrides=[f"dataset_name={DATASET}", f"hardware={HARDWARE}"],
)
nb2_dir = Path(cfg.paths.outputs_dir) / NB2_DIR_NAME
out_dir = Path(cfg.paths.outputs_dir) / "nb3"
out_dir.mkdir(parents=True, exist_ok=True)

train_manifest_path = nb2_dir / "train_manifest.json"
test_manifest_path = nb2_dir / "test_manifest.json"
if not train_manifest_path.exists() or not test_manifest_path.exists():
    raise FileNotFoundError(
        f"Missing NB2 export under {nb2_dir}. "
        "Run Notebook 2 first (train_manifest.json / test_manifest.json)."
    )

train_manifest = load_manifest(train_manifest_path)
test_manifest = load_manifest(test_manifest_path)
device = str(cfg.hardware.get("device", "cpu"))

print(f"Dataset:   {cfg.dataset_name}")
print(f"Hardware:  {cfg.hardware.name}  device={device}")
print(f"NB2:       {nb2_dir}")
print(f"NB3 out:   {out_dir}")
print(f"Train rows: {len(train_manifest)}  Test rows: {len(test_manifest)}")
print(f"Classes:   {CLASS_NAMES}")
print(f"Max scene backgrounds in train: {MAX_SCENE_TRAIN}")
print(f"Train:     {MODEL}  epochs={EPOCHS}  imgsz={IMGSZ}  batch={BATCH}")


---
## 2. Build three YOLO datasets

| Run | Train images | Val / test |
|-----|--------------|------------|
| **real_only** | real train (≤`MAX_SCENE_TRAIN` scenes) | real test (Mapillary GT) |
| **real_synth_50** | real train + **50%** of usable NB2 synth **per rare class** | same real test |
| **real_synth_100** | real train + **100%** of usable NB2 synth per rare class | same real test |

Caps are computed from the train manifest (not hardcoded 50/100), so a run that only accepted ~49 cones and ~95 bins uses ~24 / ~49 cones and ~47 / ~95 bins.

Scene backgrounds are capped so rares are not drowned (~150 scenes vs whatever synth you got).
Test never includes synthetic images or YOLO-World-only labels as GT.


In [ ]:
from edgecase_synthesis.detection_train import (
    build_yolo_dataset,
    count_usable_synthetic,
    synth_caps_for_fraction,
)

ds_root = out_dir / "datasets"

# Caps track whatever Notebook 2 actually accepted (usable = has a target box).
available_synth = count_usable_synthetic(
    train_manifest,
    class_names=CLASS_NAMES,
    aliases=LABEL_ALIASES,
)
caps_half = synth_caps_for_fraction(
    train_manifest,
    fraction=0.5,
    class_names=CLASS_NAMES,
    aliases=LABEL_ALIASES,
)
caps_full = synth_caps_for_fraction(
    train_manifest,
    fraction=1.0,
    class_names=CLASS_NAMES,
    aliases=LABEL_ALIASES,
)
print("usable NB2 synth per class:", available_synth)
print("real_synth_50 caps (50%):", caps_half)
print("real_synth_100 caps (100%):", caps_full)

yaml_real, tr_real, va_real = build_yolo_dataset(
    train_manifest=train_manifest,
    test_manifest=test_manifest,
    out_dir=ds_root / "real_only",
    class_names=CLASS_NAMES,
    aliases=LABEL_ALIASES,
    include_synthetic=False,
    max_scene_images=MAX_SCENE_TRAIN,
    scene_sample_seed=SEED,
    dataset_name="real_only",
)
yaml_s50, tr_s50, va_s50 = build_yolo_dataset(
    train_manifest=train_manifest,
    test_manifest=test_manifest,
    out_dir=ds_root / "real_synth_50",
    class_names=CLASS_NAMES,
    aliases=LABEL_ALIASES,
    include_synthetic=True,
    synthetic_fraction=0.5,
    max_scene_images=MAX_SCENE_TRAIN,
    scene_sample_seed=SEED,
    dataset_name="real_synth_50",
)
yaml_s100, tr_s100, va_s100 = build_yolo_dataset(
    train_manifest=train_manifest,
    test_manifest=test_manifest,
    out_dir=ds_root / "real_synth_100",
    class_names=CLASS_NAMES,
    aliases=LABEL_ALIASES,
    include_synthetic=True,
    synthetic_fraction=1.0,
    max_scene_images=MAX_SCENE_TRAIN,
    scene_sample_seed=SEED,
    dataset_name="real_synth_100",
)


def _print_stats(title, train_s, val_s):
    print(title)
    print(
        f"  train: images={train_s.n_images}  real={train_s.n_real}  "
        f"synth={train_s.n_synthetic}  scene={train_s.n_scene}  "
        f"(dropped {train_s.n_scene_dropped})  boxes={train_s.n_boxes}  "
        f"per_class={train_s.boxes_per_class}"
    )
    print(
        f"  val:   images={val_s.n_images}  boxes={val_s.n_boxes}  "
        f"per_class={val_s.boxes_per_class}"
    )
    if train_s.n_skipped_boxes:
        print(f"  skipped non-target / empty-synth (train): {train_s.n_skipped_boxes}")


_print_stats("real_only", tr_real, va_real)
_print_stats("real_synth_50", tr_s50, va_s50)
_print_stats("real_synth_100", tr_s100, va_s100)
print("data yaml:", yaml_real)
print("data yaml:", yaml_s50)
print("data yaml:", yaml_s100)


---
## 3. Fine-tune A: real only

Same hyperparameters for all runs so the only difference is the training set.


In [ ]:
from edgecase_synthesis.detection_train import train_detector

runs_dir = out_dir / "runs"
run_real = train_detector(
    yaml_real,
    name="real_only",
    project_dir=runs_dir,
    model_name=MODEL,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=device,
    seed=SEED,
    patience=PATIENCE,
)
print("best weights:", run_real.weights)
print("metrics:", run_real.metrics)


---
## 4. Fine-tune B: real + 50% of accepted synth


In [ ]:
run_synth_50 = train_detector(
    yaml_s50,
    name="real_synth_50",
    project_dir=runs_dir,
    model_name=MODEL,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=device,
    seed=SEED,
    patience=PATIENCE,
)
print("best weights:", run_synth_50.weights)
print("metrics:", run_synth_50.metrics)


---
## 5. Fine-tune C: real + 100% of accepted synth


In [ ]:
run_synth_100 = train_detector(
    yaml_s100,
    name="real_synth_100",
    project_dir=runs_dir,
    model_name=MODEL,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=device,
    seed=SEED,
    patience=PATIENCE,
)
print("best weights:", run_synth_100.weights)
print("metrics:", run_synth_100.metrics)


---
## 6. Compare on held-out real test

Look at **per-class AP50** for the rare classes — overall mAP can hide a tail-class win (or loss).


In [ ]:
import matplotlib.pyplot as plt
from edgecase_synthesis.detection_train import metrics_table, plot_map_comparison
from edgecase_synthesis.eda import write_json

runs = [run_real, run_synth_50, run_synth_100]
table = metrics_table(runs)
print(f"{'run':12s}  {'mAP50':>7s}  {'mAP50-95':>8s}", end="")
for cls in CLASS_NAMES:
    print(f"  {('AP50 '+cls):>16s}", end="")
print()
for row in table:
    print(
        f"{row['run']:12s}  {float(row['map50'] or 0):7.3f}  {float(row['map50_95'] or 0):8.3f}",
        end="",
    )
    for cls in CLASS_NAMES:
        print(f"  {float(row.get(f'ap50_{cls}') or 0):16.3f}", end="")
    print()

write_json(out_dir / "comparison.json", table)
fig, _ = plot_map_comparison(runs, title="Real-only vs +50 vs +100 synth (held-out real test)")
fig.savefig(out_dir / "comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved:", out_dir / "comparison.png")

---
## 7. Qualitative gallery

Side-by-side predictions on a few **real test** images.


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from edgecase_synthesis.detection_train import predict_gallery

# Prefer rare-tagged real test images for the gallery.
rare_test = [
    Path(r["path"])
    for r in test_manifest
    if str(r.get("tag", "")) in ("traffic_cone", "trash_bin")
]
scene_test = [
    Path(r["path"]) for r in test_manifest if str(r.get("tag", "")) == "scene"
]
gallery_paths = (rare_test + scene_test)[:8]
print(f"gallery images: {len(gallery_paths)}")

gal_real = predict_gallery(
    run_real.weights, gallery_paths, out_dir=out_dir / "gallery_real_only", device=device
)
gal_s50 = predict_gallery(
    run_synth_50.weights, gallery_paths, out_dir=out_dir / "gallery_real_synth_50", device=device
)
gal_s100 = predict_gallery(
    run_synth_100.weights, gallery_paths, out_dir=out_dir / "gallery_real_synth_100", device=device
)

n = min(len(gallery_paths), len(gal_real), len(gal_s50), len(gal_s100))
if n:
    fig, axes = plt.subplots(n, 3, figsize=(14, 3.5 * n))
    if n == 1:
        axes = axes.reshape(1, -1)
    for i in range(n):
        axes[i, 0].imshow(Image.open(gal_real[i]))
        axes[i, 0].set_title("real_only")
        axes[i, 0].axis("off")
        axes[i, 1].imshow(Image.open(gal_s50[i]))
        axes[i, 1].set_title("real_synth_50")
        axes[i, 1].axis("off")
        axes[i, 2].imshow(Image.open(gal_s100[i]))
        axes[i, 2].set_title("real_synth_100")
        axes[i, 2].axis("off")
    plt.tight_layout()
    gallery_path = out_dir / "gallery_compare.png"
    fig.savefig(gallery_path, dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved:", gallery_path)
else:
    print("No gallery images produced.")


---
## Wrap-up

| If you see… | Likely takeaway |
|-------------|-----------------|
| **real_synth_50 / real_synth_100 AP ↑** on rares | Accepted synth helped the tail |
| **flat / ↓** | Labels noisy, too few epochs, or rares already easy after real fine-tune |
| **scene false positives ↑** | Synth inserted objects the judge liked but the detector overfits |

Artifacts: `outputs/<dataset>/nb3/` (`datasets/`, `runs/`, `comparison.json`, galleries).

Next ideas (optional): longer schedule, stronger filter on synth boxes, or a third run with **manual** labels only on synth.